# FRED Economic Indicators for Austin Real Estate Market

This notebook fetches economic data from the Federal Reserve Economic Data (FRED) API and generates 8 charts:

1. **Austin Employment - Office Sectors** (Professional, Financial, Government, Tech)
2. **Austin Employment - Industrial** (Trade & Transportation)
3. **Austin Employment - Retail** (Leisure & Hospitality)
4. **Austin vs National Tech Employment Growth** (indexed comparison)
5. **Austin Population Growth**
6. **Austin vs National Wage Growth** (indexed comparison)
7. **Interest Rates** (10-Year Treasury vs 30-Year Mortgage)
8. **Inflation** (Core CPI vs Rent CPI, indexed comparison)

All charts use Aquila brand styling and are saved to the `charts/` directory.

## Setup and Helper Functions

In [1]:
import os
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from aquila_graphing_tools import aquila_styled_line_chart, AQUILA_COLORS, AQUILA_FONT

# Load environment variables
load_dotenv('aquila_graph.env')
fred_api_key = os.getenv('FRED_API_KEY')

if not fred_api_key:
    raise ValueError("FRED_API_KEY not found in aquila_graph.env")

print("✓ Environment loaded successfully")

✓ Environment loaded successfully


In [2]:
def fetch_fred_series(series_id, series_name=None):
    """
    Fetch FRED data and return as DataFrame with date and value columns.
    
    Parameters
    ----------
    series_id : str
        FRED series identifier
    series_name : str, optional
        Name for the value column (defaults to series_id)
    
    Returns
    -------
    pd.DataFrame
        DataFrame with 'date' and series_name columns
    """
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": fred_api_key,
        "file_type": "json"
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        observations = data.get("observations", [])
        
        if not observations:
            print(f"Warning: No data returned for series {series_id}")
            return pd.DataFrame()
        
        df = pd.DataFrame(observations)
        df['date'] = pd.to_datetime(df['date'])
        
        # Convert value to numeric, handling '.' as NaN
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        
        # Rename value column to series name
        column_name = series_name if series_name else series_id
        df = df[['date', 'value']].rename(columns={'value': column_name})
        
        # Drop NaN values
        df = df.dropna()
        
        print(f"✓ Fetched {len(df)} observations for {series_id} ({column_name})")
        return df
        
    except Exception as e:
        print(f"Error fetching series {series_id}: {str(e)}")
        return pd.DataFrame()

print("✓ Helper function defined")

✓ Helper function defined


## Chart 1: Austin Employment - Office Sectors

Tracks employment in office-related sectors:
- Professional & Business Services
- Financial Activities
- Government
- Information (Tech)

In [10]:
# Fetch employment data for office sectors
df_prof = fetch_fred_series('AUST448PBSV', 'Professional & Business Services')
df_fire = fetch_fred_series('AUST448FIRE', 'Financial Activities')
df_govt = fetch_fred_series('AUST448GOVT', 'Government')
df_info = fetch_fred_series('AUST448INFO', 'Information (Tech)')

# Only use data after 2018
df_prof = df_prof[df_prof['date'] >= '2018-01-01']
df_fire = df_fire[df_fire['date'] >= '2018-01-01']
df_govt = df_govt[df_govt['date'] >= '2018-01-01']
df_info = df_info[df_info['date'] >= '2018-01-01']

# Merge all series on date
df_office = df_prof.merge(df_fire, on='date', how='outer') \
                   .merge(df_govt, on='date', how='outer') \
                   .merge(df_info, on='date', how='outer')

# Index all series to start at 100 (first available value in each column)
indexed = df_office.copy()
for col in ['Professional & Business Services', 'Financial Activities', 'Government', 'Information (Tech)']:
    first_val = indexed[col].dropna().iloc[0]
    indexed[col] = (indexed[col] / first_val) * 100

# Convert to long format for plotting
df_office_long = indexed.melt(
    id_vars=['date'],
    var_name='Sector',
    value_name='Employment Index (Jan 2019 = 100)'
)

# Create chart
fig = aquila_styled_line_chart(
    df_office_long,
    x='date',
    y='Employment Index (Jan 2019 = 100)',
    color='Sector',
    title='Austin Employment - Office Sectors (Indexed, 2019=100)',
    height=800
)

fig.update_yaxes(rangemode='tozero', title='Indexed Employment (Jan 2019 = 100)')

# Save chart
fig.write_html('charts/austin_employment_office_sectors.html')
print("✓ Chart saved: austin_employment_office_sectors.html")

fig.show()

✓ Fetched 431 observations for AUST448PBSV (Professional & Business Services)
✓ Fetched 431 observations for AUST448FIRE (Financial Activities)
✓ Fetched 431 observations for AUST448GOVT (Government)
✓ Fetched 431 observations for AUST448INFO (Information (Tech))
✓ Chart saved: austin_employment_office_sectors.html


## Chart 2: Austin Employment - Industrial Sector

Tracks employment in Trade, Transportation & Utilities (industrial-related employment)

In [12]:
# Fetch industrial employment data
df_industrial = fetch_fred_series('AUST448TRAD', 'Trade, Transportation & Utilities')

# Only keep data from 2018 onward
df_industrial = df_industrial[df_industrial['date'] >= '2018-01-01']

# Create chart
fig = aquila_styled_line_chart(
    df_industrial,
    x='date',
    y='Trade, Transportation & Utilities',
    title='Austin Employment - Industrial Sector',
    height=800
)

fig.update_yaxes(rangemode='tozero', title='Employment (thousands)')

# Save chart
fig.write_html('charts/austin_employment_industrial.html')
print("✓ Chart saved: austin_employment_industrial.html")

fig.show()

✓ Fetched 431 observations for AUST448TRAD (Trade, Transportation & Utilities)
✓ Chart saved: austin_employment_industrial.html


## Chart 3: Austin Employment - Retail Sector

Tracks employment in Leisure & Hospitality (retail-related employment)

In [13]:
# Fetch retail employment data
df_retail = fetch_fred_series('AUST448LEIH', 'Leisure & Hospitality')
df_retail = df_retail[df_retail['date'] >= '2018-01-01']

# Create chart
fig = aquila_styled_line_chart(
    df_retail,
    x='date',
    y='Leisure & Hospitality',
    title='Austin Employment - Retail Sector',
    height=800
)

fig.update_yaxes(rangemode='tozero', title='Employment (thousands)')

# Save chart
fig.write_html('charts/austin_employment_retail.html')
print("✓ Chart saved: austin_employment_retail.html")

fig.show()

✓ Fetched 431 observations for AUST448LEIH (Leisure & Hospitality)
✓ Chart saved: austin_employment_retail.html


## Chart 4: Austin vs National Tech Employment Growth

Compares tech employment growth between Austin and the nation, indexed to 100 at the earliest common date

In [15]:
# Fetch tech employment data
df_austin_tech = fetch_fred_series('AUST448INFO', 'Austin Tech')
df_national_tech = fetch_fred_series('USINFO', 'National Tech')

# Filter both series to start after 2017
df_austin_tech = df_austin_tech[df_austin_tech['date'] >= '2018-01-01']
df_national_tech = df_national_tech[df_national_tech['date'] >= '2018-01-01']

# Merge on date
df_tech = df_austin_tech.merge(df_national_tech, on='date', how='inner')

# Find earliest common date and index both series to 100
if len(df_tech) > 0:
    base_austin = df_tech['Austin Tech'].iloc[0]
    base_national = df_tech['National Tech'].iloc[0]
    
    df_tech['Austin Tech (Index)'] = (df_tech['Austin Tech'] / base_austin) * 100
    df_tech['National Tech (Index)'] = (df_tech['National Tech'] / base_national) * 100
    
    # Convert to long format
    df_tech_long = df_tech[['date', 'Austin Tech (Index)', 'National Tech (Index)']].melt(
        id_vars=['date'],
        var_name='Region',
        value_name='Employment Index (Base 100)'
    )
    
    # Create chart
    fig = aquila_styled_line_chart(
        df_tech_long,
        x='date',
        y='Employment Index (Base 100)',
        color='Region',
        title='Austin vs National Tech Employment Growth',
        height=800
    )
    
    fig.update_yaxes(rangemode='tozero')
    
    # Save chart
    fig.write_html('charts/austin_vs_national_tech_employment.html')
    print("✓ Chart saved: austin_vs_national_tech_employment.html")
    
    fig.show()
else:
    print("Warning: No overlapping data for tech employment comparison")

✓ Fetched 431 observations for AUST448INFO (Austin Tech)
✓ Fetched 1044 observations for USINFO (National Tech)
✓ Chart saved: austin_vs_national_tech_employment.html


## Chart 6: Austin vs National Wage Growth

Compares wage growth between Austin and the nation, indexed to 100 at the earliest common date.
National data (hourly) is converted to weekly by multiplying by 40 for comparison.

In [21]:
# Fetch wage data
df_austin_wage = fetch_fred_series('SMU48124200500000003', 'Austin Hourly Wage')
df_dallas_wage = fetch_fred_series('SMU48191000500000003', 'Dallas Hourly Wage')
df_national_wage = fetch_fred_series('CES0500000003', 'National Hourly Wage')

# Convert national hourly to weekly (×40 hours) -- if desired for fair comparison, otherwise keep as hourly
df_national_wage['National Hourly Wage'] = df_national_wage['National Hourly Wage']
df_national_wage = df_national_wage[['date', 'National Hourly Wage']]

# Merge on date (inner join all three)
df_wage = df_austin_wage.merge(df_dallas_wage, on='date', how='inner') \
                        .merge(df_national_wage, on='date', how='inner')

# Index all three series to 100 at earliest common date
if len(df_wage) > 0:
    base_austin = df_wage['Austin Hourly Wage'].iloc[0]
    base_dallas = df_wage['Dallas Hourly Wage'].iloc[0]
    base_national = df_wage['National Hourly Wage'].iloc[0]
    
    df_wage['Austin Wage (Index)'] = (df_wage['Austin Hourly Wage'] / base_austin) * 100
    df_wage['Dallas Wage (Index)'] = (df_wage['Dallas Hourly Wage'] / base_dallas) * 100
    df_wage['National Wage (Index)'] = (df_wage['National Hourly Wage'] / base_national) * 100
    
    # Convert to long format
    df_wage_long = df_wage[['date', 'Austin Wage (Index)', 'Dallas Wage (Index)', 'National Wage (Index)']].melt(
        id_vars=['date'],
        var_name='Region',
        value_name='Wage Index (Base 100)'
    )
    
    # Create chart
    fig = aquila_styled_line_chart(
        df_wage_long,
        x='date',
        y='Wage Index (Base 100)',
        color='Region',
        title='Austin vs Dallas vs National Wage Growth',
        height=800
    )
        
    # Save chart
    fig.write_html('charts/austin_dallas_vs_national_wage_growth.html')
    print("✓ Chart saved: austin_dallas_vs_national_wage_growth.html")
    
    fig.show()
else:
    print("Warning: No overlapping data for wage comparison")

✓ Fetched 227 observations for SMU48124200500000003 (Austin Hourly Wage)
✓ Fetched 227 observations for SMU48191000500000003 (Dallas Hourly Wage)
✓ Fetched 238 observations for CES0500000003 (National Hourly Wage)
✓ Chart saved: austin_dallas_vs_national_wage_growth.html


## Chart 7: Interest Rates - 10-Year Treasury vs 30-Year Mortgage

Tracks two key interest rates that impact commercial real estate financing

In [24]:
# Fetch interest rate data
df_treasury = fetch_fred_series('DGS10', '10-Year Treasury')
df_mortgage = fetch_fred_series('MORTGAGE30US', '30-Year Mortgage')

# Ensure no duplicate date columns and clarify column names
if '10-Year Treasury' not in df_treasury.columns:
    df_treasury = df_treasury.rename(columns={df_treasury.columns[-1]: '10-Year Treasury'})
if '30-Year Mortgage' not in df_mortgage.columns:
    df_mortgage = df_mortgage.rename(columns={df_mortgage.columns[-1]: '30-Year Mortgage'})

# --- Make x range start from the same point (truncate to common date range) ---
# Find max of min dates for both series to ensure both have data from same start date
min_date_treasury = df_treasury['date'].min()
min_date_mortgage = df_mortgage['date'].min()
common_start_date = max(min_date_treasury, min_date_mortgage)

# Filter both DataFrames to start from the common_start_date
df_treasury_common = df_treasury[df_treasury['date'] >= common_start_date].reset_index(drop=True)
df_mortgage_common = df_mortgage[df_mortgage['date'] >= common_start_date].reset_index(drop=True)

# Merge on date - use inner join to keep only overlapping dates (makes x range match)
df_rates = df_treasury_common.merge(df_mortgage_common, on='date', how='inner')

# Remove rows where both rates are missing (i.e. drop NA rows where both columns are nan)
if '10-Year Treasury' in df_rates.columns and '30-Year Mortgage' in df_rates.columns:
    df_rates = df_rates.dropna(subset=['10-Year Treasury', '30-Year Mortgage'], how='all')
else:
    print("Warning: Missing expected columns in merged interest rate data")

# Check if data is present for both columns
if df_rates['30-Year Mortgage'].notnull().sum() == 0:
    print("Warning: No data for 30-Year Mortgage was found after merge.")

# Convert to long format, ensuring both columns appear
df_rates_long = df_rates.melt(
    id_vars=['date'],
    value_vars=['10-Year Treasury', '30-Year Mortgage'],
    var_name='Rate Type',
    value_name='Interest Rate (%)'
)

# Remove rows with missing values (so only actual rates are shown)
df_rates_long = df_rates_long.dropna(subset=['Interest Rate (%)'])

# Create chart
fig = aquila_styled_line_chart(
    df_rates_long,
    x='date',
    y='Interest Rate (%)',
    color='Rate Type',
    title='Interest Rates - Treasury & Mortgage',
    height=800
)

fig.update_yaxes(rangemode='tozero')

# Save chart
fig.write_html('charts/interest_rates_treasury_mortgage.html')
print("✓ Chart saved: interest_rates_treasury_mortgage.html")

fig.show()

✓ Fetched 15996 observations for DGS10 (10-Year Treasury)
✓ Fetched 2860 observations for MORTGAGE30US (30-Year Mortgage)
✓ Chart saved: interest_rates_treasury_mortgage.html


## Chart 8: Inflation - Core CPI vs Rent CPI

Compares general inflation (Core CPI) with rent inflation, indexed to 100 at earliest common date

In [28]:
# Fetch inflation data
df_core_cpi = fetch_fred_series('CPILFESL', 'Core CPI')
df_rent_cpi = fetch_fred_series('CUUR0000SEHC', 'Rent CPI')
df_office_ppi = fetch_fred_series('WPU43110101', 'Producer Price Index - Office Rents')
df_office_construction_ppi = fetch_fred_series('PCU236223236223', 'Producer Price Index - New Office Construction')

# Merge on date (inner merge to get overlap for all four series)
df_inflation = df_core_cpi.merge(df_rent_cpi, on='date', how='inner')
df_inflation = df_inflation.merge(df_office_ppi, on='date', how='inner')
df_inflation = df_inflation.merge(df_office_construction_ppi, on='date', how='inner')

# Index all series to 100 at earliest common date
if len(df_inflation) > 0:
    base_core = df_inflation['Core CPI'].iloc[0]
    base_rent = df_inflation['Rent CPI'].iloc[0]
    base_office = df_inflation['Producer Price Index - Office Rents'].iloc[0]
    base_office_constr = df_inflation['Producer Price Index - New Office Construction'].iloc[0]
    
    df_inflation['Core CPI (Index)'] = (df_inflation['Core CPI'] / base_core) * 100
    df_inflation['Rent CPI (Index)'] = (df_inflation['Rent CPI'] / base_rent) * 100
    df_inflation['PPI - Office Rents (Index)'] = (df_inflation['Producer Price Index - Office Rents'] / base_office) * 100
    df_inflation['PPI - New Office Construction (Index)'] = (df_inflation['Producer Price Index - New Office Construction'] / base_office_constr) * 100

    # Convert to long format for all four indices
    df_inflation_long = df_inflation[['date', 'Core CPI (Index)', 'Rent CPI (Index)', 'PPI - Office Rents (Index)', 'PPI - New Office Construction (Index)']].melt(
        id_vars=['date'],
        var_name='Inflation Type',
        value_name='Index (Base 100)'
    )
    
    # Create chart
    fig = aquila_styled_line_chart(
        df_inflation_long,
        x='date',
        y='Index (Base 100)',
        color='Inflation Type',
        title='Inflation - Core CPI, Rent CPI, Producer Price Index (Office Rents & Office Construction)',
        height=800
    )
        
    # Save chart
    fig.write_html('charts/inflation_core_rent_officeppi_officeconstr.html')
    print("✓ Chart saved: inflation_core_rent_officeppi_officeconstr.html")
    
    fig.show()
else:
    print("Warning: No overlapping data for inflation/office PPI/office construction PPI comparison")

✓ Fetched 827 observations for CPILFESL (Core CPI)
✓ Fetched 516 observations for CUUR0000SEHC (Rent CPI)
✓ Fetched 204 observations for WPU43110101 (Producer Price Index - Office Rents)
✓ Fetched 234 observations for PCU236223236223 (Producer Price Index - New Office Construction)
✓ Chart saved: inflation_core_rent_officeppi_officeconstr.html


## Summary

All 8 charts have been generated and saved to the `charts/` directory:

1. ✓ `austin_employment_office_sectors.html`
2. ✓ `austin_employment_industrial.html`
3. ✓ `austin_employment_retail.html`
4. ✓ `austin_vs_national_tech_employment.html`
5. ✓ `austin_population_growth.html`
6. ✓ `austin_vs_national_wage_growth.html`
7. ✓ `interest_rates_treasury_mortgage.html`
8. ✓ `inflation_core_vs_rent_cpi.html`

These charts are ready to be published to GitHub Pages and linked in README.md.